<a href="https://colab.research.google.com/github/bibikkkka/Big-Data_6405_FilippovAS/blob/LR2/Big_Data_6405_FilippovAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 1. Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

In [12]:
!pip install pyspark

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, explode, split, lower, trim, regexp_replace, count, rank
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Programming Languages Report") \
    .master("local[*]") \
    .getOrCreate()

In [14]:
langs_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("programming-languages.csv")

print("=== Данные о языках программирования ===")
langs_df.show(10, truncate=False)

programming_languages = langs_df.select("name") \
    .rdd \
    .map(lambda x: x[0].lower()) \
    .collect()

=== Данные о языках программирования ===
+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
|ABSYS     |https://en.wikipedia.org/wiki/ABSYS                      |
+----------+------------------------

In [15]:
text_rdd = spark.sparkContext.textFile("posts_sample.xml")
rows_rdd = text_rdd.filter(lambda line: "<row " in line)

print("\n=== Пример XML данных ===")
for i, line in enumerate(text_rdd.take(5)):
    print(f"Строка {i+1}: {line[:100]}...")


=== Пример XML данных ===
Строка 1: <?xml version="1.0" encoding="utf-8"?>...
Строка 2: <posts>...
Строка 3:   <row Id="4" PostTypeId="1" AcceptedAnswerId="7" CreationDate="2008-07-31T21:42:52.667" Score="630"...
Строка 4:   <row Id="6" PostTypeId="1" AcceptedAnswerId="31" CreationDate="2008-07-31T22:08:08.620" Score="281...
Строка 5:   <row Id="7" PostTypeId="2" ParentId="4" CreationDate="2008-07-31T22:17:57.883" Score="425" Body="&...


In [16]:
from xml.etree import ElementTree

def parse_xml_row(line):
    xml_str = f"<root>{line}</root>"
    root = ElementTree.fromstring(xml_str)
    row = root.find("row")
    creation_date = row.get("CreationDate")
    tags = row.get("Tags")
    return (creation_date, tags) if creation_date and tags else (None, None)

parsed_rdd = rows_rdd.map(parse_xml_row).filter(lambda x: x[0] is not None and x[1] is not None)
posts_df = spark.createDataFrame(parsed_rdd, ["CreationDate", "Tags"])

In [17]:
processed_df = posts_df \
    .withColumn("Year", substring(col("CreationDate"), 1, 4)) \
    .withColumn("Tag", explode(split(regexp_replace(col("Tags"), "[<>]", " "), " "))) \
    .filter(trim(col("Tag")) != "") \
    .withColumn("Tag", lower(trim(col("Tag")))) \
    .filter(col("Tag").isin(programming_languages)) \
    .select("Year", "Tag")

In [18]:
window_spec = Window.partitionBy("Year").orderBy(F.desc("Count"))

report_df = processed_df \
    .filter(col("Year").between("2010", "2020")) \
    .groupBy("Year", "Tag") \
    .agg(count("*").alias("Count")) \
    .withColumn("Rank", rank().over(window_spec)) \
    .filter(col("Rank") <= 10) \
    .select("Year", "Tag", "Count") \
    .orderBy("Year", F.desc("Count"))

print("\n=== Итоговый отчет ===")
report_df.show(100)

report_df.write.parquet("report", mode="overwrite")
spark.stop()


=== Итоговый отчет ===
+----+-----------+-----+
|Year|        Tag|Count|
+----+-----------+-----+
|2010|       java|   52|
|2010|        php|   46|
|2010| javascript|   44|
|2010|     python|   26|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   12|
|2010|     delphi|    8|
|2010|          r|    3|
|2010|applescript|    3|
|2010|       perl|    3|
|2010|       bash|    3|
|2011|        php|  102|
|2011|       java|   93|
|2011| javascript|   83|
|2011|     python|   37|
|2011|objective-c|   34|
|2011|          c|   24|
|2011|       ruby|   20|
|2011|       perl|    9|
|2011|     delphi|    8|
|2011|       bash|    7|
|2012|        php|  154|
|2012| javascript|  132|
|2012|       java|  124|
|2012|     python|   69|
|2012|objective-c|   45|
|2012|          c|   27|
|2012|       ruby|   27|
|2012|       bash|   10|
|2012|          r|    9|
|2012|      xpath|    6|
|2012|     matlab|    6|
|2012|        lua|    6|
|2012|      scala|    6|
|2013| javascript|  198|
|